In [ ]:
!pip install category_encoders
!pip install xgboost
!pip install lightgbm
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.5 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import joblib
import warnings
warnings.filterwarnings("ignore")

In [ ]:
X_train = joblib.load("X_train_enc.pkl")
X_val   = joblib.load("X_val_enc.pkl")
y_train = joblib.load("y_train.pkl")
y_val   = joblib.load("y_val.pkl")

In [ ]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "ElasticNet": ElasticNet(),
    "DecisionTree": DecisionTreeRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "XGBoost": XGBRegressor(verbosity=0),
    "LightGBM": LGBMRegressor(),
    "CatBoost": CatBoostRegressor(verbose=0)
}

In [ ]:
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006541 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 280
[LightGBM] [Info] Number of data points in the train set: 22756, number of used features: 34
[LightGBM] [Info] Start training from score 130384.922438


In [ ]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_val)

    results.append({
        "Model": name,
        "R2": r2_score(y_val, y_pred),
        "MAE": mean_absolute_error(y_val, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_val, y_pred))
    })

results_df = pd.DataFrame(results).sort_values(by="R2", ascending=False)
results_df

,Model,R2,MAE,RMSE
10,CatBoost,0.912390,14110.192060,22800.381572
8,XGBoost,0.908616,14406.640625,23286.379882
9,LightGBM,0.904566,15025.668487,23796.755137
5,RandomForest,0.903409,14155.458338,23940.607585
6,ExtraTrees,0.889719,15340.528780,25580.955631
7,GradientBoosting,0.871553,18202.662192,27607.572037
4,DecisionTree,0.823017,18949.812930,32406.436918
2,Lasso,0.779471,26277.710792,36174.201018
0,LinearRegression,0.779470,26278.492744,36174.276137
1,Ridge,0.779469,26278.364081,36174.347478


In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Meilleur modèle :", best_model_name)

Meilleur modèle : CatBoost


In [ ]:
joblib.dump(best_model, "best_model.pkl")
joblib.dump(best_model_name, "best_model_name.pkl")

['best_model_name.pkl']

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = []

for name, model in models.items():
    r2_scores, mae_scores, rmse_scores = [], [], []

    for train_idx, test_idx in kf.split(X_train):
        X_tr, X_te = X_train.iloc[train_idx], X_train.iloc[test_idx]
        y_tr, y_te = y_train.iloc[train_idx], y_train.iloc[test_idx]

        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        r2_scores.append(r2_score(y_te, y_pred))
        mae_scores.append(mean_absolute_error(y_te, y_pred))
        rmse_scores.append(np.sqrt(mean_squared_error(y_te, y_pred)))

    cv_results.append({
        "Model": name,
        "R2_CV_mean": np.mean(r2_scores),
        "MAE_CV_mean": np.mean(mae_scores),
        "RMSE_CV_mean": np.mean(rmse_scores)
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(by="R2_CV_mean", ascending=False)
cv_results_df

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005465 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 280
[LightGBM] [Info] Number of data points in the train set: 18204, number of used features: 34
[LightGBM] [Info] Start training from score 130156.885300
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005548 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 280
[LightGBM] [Info] Number of data points in the train set: 18205, number of used features: 34
[LightGBM] [Info] Start training from score 130040.906509
[LightGBM] [Warning]

,Model,R2_CV_mean,MAE_CV_mean,RMSE_CV_mean
10,CatBoost,0.900544,14515.137482,24211.889188
8,XGBoost,0.896643,14753.772852,24680.617614
9,LightGBM,0.894137,15332.207840,24980.326391
5,RandomForest,0.893439,14513.827253,25061.443108
6,ExtraTrees,0.878692,15678.704183,26741.561270
7,GradientBoosting,0.864485,18376.808026,28260.033490
4,DecisionTree,0.794825,19687.889713,34779.076405
1,Ridge,0.770293,26226.717322,36798.722995
2,Lasso,0.770292,26226.475614,36798.803490
0,LinearRegression,0.770292,26226.917374,36798.828901


In [ ]:
best_model_name = cv_results_df.iloc[0]["Model"]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)

print("Meilleur modèle (CV) :", best_model_name)

Meilleur modèle (CV) : CatBoost


In [ ]:
y_val_pred = best_model.predict(X_val)

print("R2 :", r2_score(y_val, y_val_pred))
print("MAE :", mean_absolute_error(y_val, y_val_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_val, y_val_pred)))

R2 : 0.9123902041623347
MAE : 14110.192060391733
RMSE : 22800.38157206818


In [ ]:
joblib.dump(best_model, "best_model.pkl")
joblib.dump(best_model_name, "best_model_name.pkl")